# Disc Detection

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Needed to import from the enderscope library

import os

os.chdir("..")

In [ ]:
%matplotlib widget

## Imports

In [ ]:
from pathlib import Path

from rich.pretty import pprint

import numpy as np
import pandas as pd
import cv2

from enderleaf.draw import image_grid
from enderleaf.tools import read_dataframe
from enderleaf.image import load_image, to_pil, canny, find_circles, filter_circles

## Constants

In [ ]:
PATH_TO_DATA = Path(".").joinpath("output", "job_data", "Exp26DM14", "I2")
PATH_TO_IMAGES = Path(".").joinpath("output", "images", "Exp26DM14", "I2")
FACTOR = 4
MAX_CIRCLES=4

## Functions

In [ ]:
def load(row):
    return load_image(PATH_TO_IMAGES.joinpath(row.file_name))

## Load Data

In [ ]:
csvs = list(PATH_TO_DATA.glob("*.csv"))
len(csvs)

In [ ]:
df = pd.concat(
    [
        read_dataframe(f)
        for f in Path(".").joinpath("output", "job_data", "Exp26DM14", "I2").glob("*.csv")
    ]
).sort_values(["job_ts", "row","col"])
df

In [ ]:
row = df.sample(n=1).iloc[0]
pprint({k: row[k] for k in ["file_name", "row", "col"]})
image = load(row)
to_pil(image)

In [ ]:
image = load(row)
im_width = image.shape[1] // FACTOR
im_height = image.shape[0] // FACTOR
image = cv2.resize(image, (im_width, im_height))
edges = canny(
    image=image, color_space="rgb", channel="blue", min_thresholf=150, max_threshold=255
)

circles = filter_circles(
    find_circles(
        edges=edges,
        radii=np.arange(450 // FACTOR, 550 // FACTOR, 20 // FACTOR),
        max_circles=MAX_CIRCLES,
    ),
    img_width=im_width,
    img_height=im_height,
)

out = image.copy()
for accu, cx, cy, r in circles["accepted"]:
    print(accu, cx, cy, r)
    out = cv2.circle(out, (cx, cy), r, (255, 0, 255), 2)
for accu, cx, cy, r in circles["discarded"]:
    print(accu, cx, cy, r)
    out = cv2.circle(out, (cx, cy), r, (255, 0, 0), 2)

image_grid([edges, out], row_count=1, col_count=2)